# Arabic legal documents: data preparation

Run locally in VS Code. Select `.venv/bin/python` as the notebook kernel and install `requirements.txt` from the terminal first. PDF rendering uses PyMuPDF and does not require Poppler.


In [ ]:
import sys  # Built-in: shows which Python interpreter runs the notebook.

print(f"Python interpreter: {sys.executable}")


## Install PDF-processing dependencies

Run this cell once in your selected notebook kernel. `%pip` installs into that kernel’s environment. Restart the kernel after installation, then run the remaining cells in order. Select the project’s `.venv/bin/python` to use the environment prepared for this project.


In [ ]:
%pip install "PyMuPDF>=1.26,<2" "Pillow>=12.2.0,<13" "tqdm>=4.67,<5"



## Start

PDFs are read from the project’s `data` folder.


In [ ]:
import os  # Built-in: provides operating-system tools for files, folders, and environment variables.
from os.path import join  # Built-in: combines folder and file names into a path.
import random  # Built-in: supports random sampling and shuffling for later data preparation.
from tqdm.auto import tqdm  # Displays progress bars when processing multiple documents or pages.

from glob import glob  # Built-in: finds files matching a pattern, such as "*.pdf".
from pathlib import Path  # Built-in: handles paths and checks whether files or folders exist.
import pymupdf  # Renders PDF pages directly into images without external tools.
from PIL import Image  # Pillow: opens, resizes, and saves images for VLM inputs.
from PIL import ImageEnhance  # Pillow: adjusts image contrast before OCR.

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
data_dir = str(project_dir / "data")

if not Path(data_dir).is_dir():
    raise FileNotFoundError(f"Data folder not found: {data_dir}. Open the project folder in VS Code.")
print(f"Data directory: {data_dir}")
pdf_files = glob(join(data_dir, "*.pdf"))
print(f"PDF files: {len(pdf_files)}")


## Hugging Face authentication (local)

This replaces Colab secrets and shell login with local `.env` loading and the Hugging Face Python API. Run the project-path cell above first. Set `HF_TOKEN` in `.env`, then run this cell to validate it and save it in the local Hugging Face authentication cache. Only the account name is printed. The token is not added to Git credentials.

This cell is needed for model access, not PDF conversion; you can skip it when preparing images only. The packages `python-dotenv` and `huggingface-hub` are installed through the project requirements (the latter is also a Transformers dependency). Gemma access may require accepting the model terms on Hugging Face.


In [ ]:
from dotenv import dotenv_values  # Reads local configuration without printing credentials.
from huggingface_hub import HfApi, login  # Authenticates with Hugging Face through its Python API.

# Read the project-specific token directly from the ignored .env file.
env_path = project_dir / ".env"
hf_token = (dotenv_values(env_path).get("HF_TOKEN") or "").strip()

if not hf_token:
    raise RuntimeError("Set HF_TOKEN in the project's .env file before running authentication.")

try:
    # Validate the token before saving it in Hugging Face's local authentication cache.
    hf_account = HfApi().whoami(token=hf_token)
    login(token=hf_token, add_to_git_credential=False)
except Exception:
    raise RuntimeError("Hugging Face login failed. Check the token and network connection.") from None

print(f"Hugging Face authentication successful: {hf_account['name']}")



## Pdf to Images

In [ ]:
def preprocess_image(image, max_width=600):
    """
    Preprocess image to reduce size and prepare for OCR.

    Args:
        image: PIL Image object
        max_width: Maximum width in pixels (height auto-calculated to maintain ratio)

    Steps:
    1. Convert to grayscale (reduces size and improves OCR)
    2. Resize to reasonable width while maintaining aspect ratio
    3. Increase contrast
    """

    # Convert to grayscale
    gray_image = image.convert('L')

    # Resize if image is too large
    if gray_image.width > max_width:
        # Calculate new height to maintain aspect ratio
        ratio = max_width / gray_image.width
        new_height = int(gray_image.height * ratio)
        gray_image = gray_image.resize((max_width, new_height), Image.LANCZOS)

    enhancer = ImageEnhance.Contrast(gray_image)
    enhanced_image = enhancer.enhance(1.5)

    return enhanced_image


In [ ]:
def convert_pdf_to_images(pdf_path, output_base_dir, max_width=600):
    """
    Render PDF pages with PyMuPDF, preprocess them, and save JPEG images.

    Args:
        pdf_path: Path to the PDF file.
        output_base_dir: Base directory to save images.
        max_width: Maximum output width in pixels (default: 600).
    """
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    output_dir = join(output_base_dir, pdf_name)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Converting {pdf_name}.pdf...")
    generated_paths = []

    # Render one page at a time to avoid loading the entire PDF into memory.
    with pymupdf.open(pdf_path) as document:
        for page_num, page in enumerate(document, start=1):
            # Render at 200 DPI in RGB; no external Poppler installation is needed.
            pixmap = page.get_pixmap(dpi=200, colorspace=pymupdf.csRGB, alpha=False)
            image = Image.frombytes("RGB", (pixmap.width, pixmap.height), pixmap.samples)
            processed_image = preprocess_image(image, max_width)

            output_path = join(output_dir, f"page_{page_num:03d}.jpg")
            processed_image.save(output_path, "JPEG", quality=85, optimize=True)
            image.close()
            processed_image.close()
            print(f"  Saved page {page_num} -> {output_path}")
            generated_paths.append(output_path)

    return generated_paths


### Test PDF conversion

This cell tests the PDF-to-image pipeline on `reconciliation_center_regulations.pdf` in the local `data/` folder. It checks that the PDF exists, then calls `convert_pdf_to_images()`.

Each page is rendered with PyMuPDF at 200 DPI, converted to grayscale, resized to a maximum width of 600 pixels while preserving its aspect ratio, and given increased contrast. The processed pages are saved as JPEG files named `page_001.jpg`, `page_002.jpg`, and so on, under `outputs/images/reconciliation_center_regulations/`. The cell returns their paths in `generated_paths` and prints the number of images created.

This checks image preparation; it does not extract text or train a model. You can change `sample_pdf_path` to test another PDF and adjust `max_width` to retain more detail. Running it again overwrites images with the same filenames.


In [ ]:
# Select an existing PDF from the local data folder.
sample_pdf_path = Path(data_dir) / "reconciliation_center_regulations.pdf"

# Store generated images inside the project instead of Colab's /content folder.
output_base_dir = project_dir / "outputs" / "images"

if not sample_pdf_path.is_file():
    raise FileNotFoundError(f"Sample PDF not found: {sample_pdf_path}")

generated_paths = convert_pdf_to_images(
    pdf_path=str(sample_pdf_path),
    output_base_dir=str(output_base_dir),
    max_width=600,  # A 200px page is usually too small to read Arabic text clearly.
)

print(f"Generated {len(generated_paths)} images in {output_base_dir / sample_pdf_path.stem}")


### List and convert all PDFs

This cell lists all PDFs in the local `data/` folder and converts every document into preprocessed page images, showing a progress bar. Images are saved under `outputs/images/<pdf_name>/` at a maximum width of 600 pixels. `all_generated_paths` stores the output paths for each document. Run the imports and function definitions above first. Running this again overwrites images with the same filenames.


In [ ]:
# Find and list every PDF in the project's data folder.
pdf_files = sorted(glob(join(data_dir, "*.pdf")))
output_dir = project_dir / "outputs" / "images"
max_width = 600

if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in {data_dir}")

print(f"Found {len(pdf_files)} PDFs:")
for pdf_file in pdf_files:
    print(f"  - {Path(pdf_file).name}")

# Convert all PDFs, keeping each document's pages in its own folder.
all_generated_paths = {}
for pdf_file in tqdm(pdf_files, desc="Converting PDFs", unit="pdf"):
    all_generated_paths[Path(pdf_file).name] = convert_pdf_to_images(
        pdf_path=pdf_file,
        output_base_dir=str(output_dir),
        max_width=max_width,
    )

image_count = sum(len(paths) for paths in all_generated_paths.values())
print(f"Generated {image_count} page images from {len(pdf_files)} PDFs.")


## Evaluate Base Model

Prepare the selected instruction-tuned checkpoint, `google/gemma-3-4b-it`, and a local page image for baseline evaluation before finetuning. The screenshot uses the pretrained `-pt` variant; this project uses the previously selected `-it` variant for instruction-based transcription.

This cell imports the processor and model classes, checks the sample image, converts it to RGB, and detects CUDA, Apple MPS, or CPU. It does not download weights, load the model, or generate text yet. `requests` is unnecessary because the input is a local image. Run the project-path and single-PDF conversion cells first.


### Install model dependencies

Run this installation cell in the active notebook kernel if Transformers or PyTorch is missing. Restart the kernel afterward and rerun the imports, project-path setup, and image preparation cells. `%pip` targets the notebook environment rather than another terminal Python installation. This installs libraries, not model weights.


In [ ]:
%pip install "transformers>=4.57,<5" "torch>=2.5,<3" "accelerate>=1,<2" "Pillow>=12.2.0,<13"


In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration  # Prepares model inputs and provides Gemma 3's vision-language model class.
from PIL import Image  # Opens the local page image for future model input.
import torch  # Provides tensors and detects available computation hardware.

# Use the instruction-tuned checkpoint selected for this project.
model_id = "google/gemma-3-4b-it"

# Select a page produced by the PDF conversion section above.
sample_image_path = project_dir / "outputs" / "images" / "reconciliation_center_regulations" / "page_001.jpg"
if not sample_image_path.is_file():
    raise FileNotFoundError("Run the single-PDF conversion cell first to create the sample image.")

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon GPU.
else:
    device = "cpu"

with Image.open(sample_image_path) as image:
    sample_image = image.convert("RGB")

print(f"Model checkpoint: {model_id}")
print(f"Available device: {device}")
print(f"Sample image: {sample_image_path.name}, size: {sample_image.size}")


### Preview the sample page

Display the selected JPEG inline to inspect text legibility before model inference. Run the base-model setup cell first; it defines and validates `sample_image_path`. This preview does not perform OCR.


In [ ]:
from IPython.display import display, Image as IPImage  # Displays a local image inline without conflicting with Pillow's Image.

# Run the base-model setup cell first to define sample_image_path.
display(IPImage(filename=str(sample_image_path)))


### Load Gemma and its processor

Run Hugging Face authentication and the model setup cells first. This cell downloads `google/gemma-3-4b-it` weights on first use and caches them locally, then loads the model in evaluation mode and its matching processor. Access requires approval of the Gemma terms for your Hugging Face account.

`dtype="auto"` uses the checkpoint precision; it does not quantize the model. `device_map="auto"` uses Accelerate to determine device placement, which is printed after loading. It does not guarantee that every layer runs on the Apple GPU. Model weights alone require several GB, and inference needs additional memory. Loading may fail if RAM or device memory is insufficient. `.eval()` does not disable gradients; generation will use `torch.inference_mode()` in a later cell. This cell does not generate text.


In [ ]:
# Download/cache the weights and let Accelerate choose their placement.
model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    dtype="auto",       # Uses the checkpoint's floating-point precision; this is not quantization.
    device_map="auto",  # Placement depends on available hardware and memory, not just the device variable.
).eval()  # Evaluation mode disables training behavior such as dropout.

# Load the image processor, tokenizer, and chat template for this checkpoint.
processor = AutoProcessor.from_pretrained(model_id)

print(f"Model loaded: {model_id}")
print(f"Model dtype: {model.dtype}")
print(f"Device placement: {getattr(model, 'hf_device_map', None)}")


### Generate a baseline transcription

Run the model-loading and image-preparation cells first. The processor packages the RGB image and a source-faithful transcription prompt into Gemma’s chat format. Greedy generation runs without gradients, and only newly generated tokens are decoded. The cell prints transcription, generation duration, and output token count, with a warning when the 1,024-token budget is reached.

Check the output against the source image: a prompt and deterministic decoding cannot eliminate hallucination. `[غير مقروء]` is the requested marker for unreadable text. This is a qualitative baseline, not a scored accuracy evaluation. The timing covers generation only, not model loading or input processing.


In [ ]:
import time  # Measures generation time for this baseline run.

prompt = """
Transcribe only the text visibly present in this image into Markdown.
Preserve the original Arabic wording, numbers, punctuation, and reading order.
Do not summarize, translate, correct, infer, or add words absent from the image.
Mark unreadable text as [غير مقروء] rather than guessing.
Use Markdown only to reflect visible document structure; do not invent headings.
Return only the transcription, without an introduction or a code fence.
""".strip()

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": sample_image},  # RGB Pillow image prepared above.
            {"type": "text", "text": prompt},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# Match image tensors to the vision encoder's precision without changing token IDs.
if "pixel_values" in inputs:
    inputs["pixel_values"] = inputs["pixel_values"].to(dtype=model.dtype)

input_len = inputs["input_ids"].shape[-1]
max_new_tokens = 1024  # Increase if dense pages reach the output limit; more tokens cost time/memory.

# Synchronize GPU work so the recorded duration includes completed generation.
def synchronize_generation_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elif model.device.type == "mps":
        torch.mps.synchronize()

synchronize_generation_device()
started_at = time.perf_counter()
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # Greedy decoding; this does not guarantee transcription fidelity.
    )
synchronize_generation_device()
generation_seconds = time.perf_counter() - started_at

# Decode only generated tokens, excluding the input prompt.
generation = output_ids[0, input_len:]
decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)
print(f"\nGeneration time: {generation_seconds:.2f}s; output tokens: {generation.numel()}")
if generation.numel() >= max_new_tokens:
    print("Warning: output reached max_new_tokens. Check whether the transcription is incomplete.")
